<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_1_Confusion_Matrix_Basic_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: Predicting Credit Default: Part 2
## The Confusion Matrix and Metrics

**Author:** Brad Sheese

---

## Introduction: Moving Beyond Accuracy

In the previous notebook, we saw that our Credit Default dataset is imbalanced (~70% good, ~30% bad). We learned that **Accuracy** can be a trap—a model that does nothing but guess "Good" for everyone is 70% accurate, but it fails to catch any defaults.

In this notebook, we dive into the tool that data scientists use to see the "full picture": the **Confusion Matrix**. We will learn how to dissect a model's successes and failures using four fundamental metrics: **Accuracy**, **Precision**, **Recall**, and **F1-Score**.

### Learning Objectives
By the end of this notebook, you will be able to:
1. Build and interpret a **Confusion Matrix**.
2. Identify **Type I (False Alarm)** vs. **Type II (Missed Case)** errors.
3. Calculate and interpret **Precision** and **Recall** in a business context.
4. Use the **F1-Score** to balance the trade-off between different types of errors.

## Problem 1: Setup and Baseline Model

We'll reload the Credit Default data and retrain our Logistic Regression model using the `class_weight='balanced'` parameter we introduced in Part 1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Setup complete. Model trained!")

## Problem 2: The Confusion Matrix (The Error Map)

The **Confusion Matrix** is a table that maps the model's predictions against the actual outcomes. It allows us to see exactly where the "confusion" is happening.

### Terminology:
- **True Positives (TP)**: We predicted Default, and they did Default. (Success!)
- **True Negatives (TN)**: We predicted Good, and they were Good. (Success!)
- **False Positives (FP)**: We predicted Default, but they were Good. (**False Alarm** / Type I Error).
- **False Negatives (FN)**: We predicted Good, but they actually Defaulted. (**Missed Case** / Type II Error).

In [ ]:
# Generate the matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"--- Confusion Matrix Breakdown ---")
print(f"Correctly Identified Good (TN): {tn}")
print(f"Correctly Identified Bad  (TP): {tp}")
print(f"False Alarms / Type I      (FP): {fp}")
print(f"Missed Defaults / Type II  (FN): {fn}")

# Visualize
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted: Good', 'Predicted: Bad'])
ax.set_yticklabels(['Actual: Good', 'Actual: Bad'])
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
ax.set_title('Confusion Matrix: Credit Default Prediction')

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=18, fontweight='bold')

plt.show()

### Look Closely:
Which error is more frequent for our model? In a credit default scenario, which mistake is usually more expensive for a bank: a False Alarm (denying a good customer) or a Missed Case (loaning money to someone who won't pay it back)?

## Problem 3: Accuracy vs. The Baseline

As we discussed, **Accuracy** is simply the total percentage of correct guesses.

$$Accuracy = \frac{TP + TN}{Total}$$

Let's calculate our model's accuracy and compare it to the "Naive Baseline" (guessing the majority class for everyone).

In [ ]:
acc = accuracy_score(y_test, y_pred)
baseline = (y_test == 0).mean()

print(f"Model Accuracy: {acc*100:.1f}%")
print(f"Baseline (Always Good): {baseline*100:.1f}%")
print(f"\nImprovement over baseline: {(acc - baseline)*100:.1f} percentage points")

## Problem 4: Precision and Recall (The Trade-Off)

To get a better sense of how the model handles the "Bad" class, we use Precision and Recall.

### Precision: How reliable are our positive hits?
When the model flags someone as a default risk, how often is it right?
$$Precision = \frac{TP}{TP + FP}$$
**High Precision** means you have very few **False Alarms**.

### Recall: How many actual cases did we catch?
Of all the people who actually defaulted, what percentage did we catch?
$$Recall = \frac{TP}{TP + FN}$$
**High Recall** means you have very few **Missed Cases**.

In [ ]:
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

print(f"Precision: {prec*100:.1f}% (Reliability of Default predictions)")
print(f"Recall:    {rec*100:.1f}% (Percentage of actual Defaults caught)")

## Problem 5: The F1-Score (The Harmonic Mean)

If you want a single number that summarizes the "balance" between Precision and Recall, use the **F1-Score**. 

The F1-Score is the **Harmonic Mean** of Precision and Recall. Unlike a standard average, the harmonic mean is very sensitive to low values. If either Precision or Recall is terrible, the F1-Score will be terrible.

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

In [ ]:
f1 = f1_score(y_test, y_pred)
print(f"F1-Score: {f1*100:.1f}%")

## Problem 6: Visualizing the Metric Trade-off

Remember the "Soft Switch" (Sigmoid) from Part 1? By default, we split at **0.5**. But what if we moved that line?

- If we move the threshold to **0.8**, we become very conservative. Precision goes UP (we only flag people we are certain about), but Recall goes DOWN (we miss more people).
- If we move the threshold to **0.2**, we become very sensitive. Recall goes UP (we catch almost everyone), but Precision goes DOWN (we have many false alarms).

Let's see this in action.

In [ ]:
thresholds = np.arange(0.05, 0.95, 0.05)
precisions = []
recalls = []
f1s = []

for t in thresholds:
    y_t = (y_proba >= t).astype(int)
    precisions.append(precision_score(y_test, y_t, zero_division=0))
    recalls.append(recall_score(y_test, y_t, zero_division=0))
    f1s.append(f1_score(y_test, y_t, zero_division=0))

plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions, label='Precision', marker='o', color='green')
plt.plot(thresholds, recalls, label='Recall', marker='s', color='blue')
plt.plot(thresholds, f1s, label='F1-Score', marker='^', color='red')

plt.axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='Default Threshold')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('How Threshold Affects Classification Metrics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### One Thing to Notice:
Where does the F1-Score peak? Is it at the default 0.5 threshold? Often, the "best" balance for a model isn't at the default setting. In Notebook 3, we'll learn a systematic way to find the "Goldilocks" threshold for your specific business problem.

## Conclusion
We've moved from simple "Guessing" to a nuanced evaluation of our model. We've learned that:
1. **Accuracy** hides errors in imbalanced data.
2. **Precision** measures reliability (avoiding false alarms).
3. **Recall** measures sensitivity (avoiding missed cases).
4. **F1-Score** looks for the middle ground.

In the next notebook, we'll look at **ROC Curves** and **AUC**, which allow us to evaluate a model's performance across *all* possible thresholds simultaneously.